# 手写 KV cache，再读 MiniMind

从空缓存开始，亲手实现 **读旧 KV → 追加新 KV → 用合并后的 KV 做 Attention → 把更新后的状态交给下一轮**。主线只依赖 PyTorch，可在你的 Miniconda `base` 内核按顺序执行。我们用小张量、两层 Attention 和六个 token 的词表，省掉模型下载和完整模型定义。

| 变量 | 谁持有 | 内容 |
| --- | --- | --- |
| `past_key_value` | 单个 Attention 层的入参 | 上一轮交给该层的 `(K, V)`，首轮是 `None` |
| `present_key_value` | 单个 Attention 层的返回值 | 旧位置和本轮新位置的 `(K, V)` |
| `past_key_values` | 模型和生成循环 | 每层一个 KV 对。下一轮把第 `i` 对交回第 `i` 层 |

**外层循环接住返回值并保存到下一轮。** 这版 Attention 层通过参数读缓存、通过返回值交出缓存。

## 先只做一层的 KV 拼接

暂时假设 K/V 已算好。按 MiniMind 的布局 `[B, T, H_kv, D_head]`，沿序列轴 `dim=1` 追加。给 K 和 V 填入容易辨认的数字，执行前先猜新旧缓存的内容。

留意两个独立决定。**传入了旧状态，就读它。** **`use_cache=True` 时，才把更新后的状态返回给调用方。** 因此在 MiniMind 的层内，传了旧状态但把 `use_cache` 设成 `False`，本轮仍会用历史，只是下轮拿不到更新后的缓存。

In [ ]:
import torch
import torch.nn.functional as F

def append_kv(k_new, v_new, past_key_value=None, *, use_cache=True):
    if past_key_value is None:
        k_all, v_all = k_new, v_new
    else:
        k_old, v_old = past_key_value
        k_all = torch.cat((k_old, k_new), dim=1)
        v_all = torch.cat((v_old, v_new), dim=1)

    present = (k_all, v_all) if use_cache else None
    return k_all, v_all, present

k_prompt = torch.tensor([[[[10., 11.]], [[20., 21.]]]])  # [1, 2, 1, 2]
v_prompt = k_prompt + 100
k_next = torch.tensor([[[[30., 31.]]]])                  # [1, 1, 1, 2]
v_next = k_next + 100

_, _, prefill_kv = append_kv(k_prompt, v_prompt)
k_all, v_all, decode_kv = append_kv(k_next, v_next, prefill_kv)
print("prefill K", prefill_kv[0][0, :, 0].tolist())
print("decode  K", k_all[0, :, 0].tolist())
print("decode  V", v_all[0, :, 0].tolist())
assert prefill_kv[0].shape == (1, 2, 1, 2)  # 拼接没有原地改动旧状态
assert decode_kv[0].shape == (1, 3, 1, 2)

same_k, same_v, no_present = append_kv(k_next, v_next, prefill_kv, use_cache=False)
torch.testing.assert_close(same_k, k_all)
torch.testing.assert_close(same_v, v_all)
assert no_present is None

## 用旧缓存长度确定新位置

连续、无 padding 的简单实现里，`past_len = K_old.shape[1]`。如果已有 4 个位置，又送来 2 个，两个新 Q 的位置是 4 和 5。这个偏移同时用于 **新 Q/K 的 RoPE** 和 **因果 mask**。旧 K 已按旧位置旋转过，拼接时直接读。V 不做 RoPE。

![续接两个 token 时，按绝对位置对齐和按左上角对齐的因果 mask](figures/kv-cache-causal-mask.svg)

下面把 RoPE 简化成每头二维向量的一次平面旋转，角度取位置乘固定频率。重点是追踪位置。对 SDPA 的布尔 mask，`True` 表示可读。

In [ ]:
def positions_and_mask(past_len, new_len, device=None):
    query_positions = torch.arange(past_len, past_len + new_len, device=device)
    key_positions = torch.arange(past_len + new_len, device=device)
    allowed = key_positions[None, :] <= query_positions[:, None]
    return query_positions, allowed

def rotate_2d(x, positions):
    angle = positions.to(x.dtype)[None, :, None] * 0.5
    a, b = x.unbind(dim=-1)
    return torch.stack(
        (a * angle.cos() - b * angle.sin(), a * angle.sin() + b * angle.cos()),
        dim=-1,
    )

new_positions, allowed = positions_and_mask(past_len=4, new_len=2)
print("新 Q/K 的位置", new_positions.tolist())
print("两行 Q 可以读取哪些 K")
print(allowed.int())
assert allowed.tolist() == [
    [True, True, True, True, True, False],
    [True, True, True, True, True, True],
]

same_vector = torch.tensor([[[[1., 0.]], [[1., 0.]]]])
rotated = rotate_2d(same_vector, new_positions)
wrong = rotate_2d(same_vector, torch.arange(2))
print("正确位置的旋转结果", rotated[0, :, 0].tolist())
assert not torch.allclose(rotated, wrong)  # 每轮都从 0 旋转会改变 K

## 把读写缓存接入一次 Attention

下面写出对应 MiniMind `Attention.forward` 的最小实现。输入 `x` 只含本轮要计算的位置，布局为 `[B, T_new, 4]`。用 **2 个 Q 头、1 个 KV 头、每头维度 2** 模拟 GQA。

1. **投影新位置**。Q 是 `[B, T_new, 2, 2]`，K/V 是 `[B, T_new, 1, 2]`。
2. **旋转新 Q/K，再拼接旧 KV**。只有 K/V 的时间轴增长到 `T_past + T_new`。
3. **先形成只有 1 个 KV 头的返回状态**，再为 Attention 临时复制 KV 头。
4. **只算新 Q 的输出**。结果仍有 `T_new` 行，与 `present` 一起返回。

示例保留投影和 Attention，省略 MiniMind 的 Q/K 归一化、输出投影和 FFN。

In [ ]:
class TinyAttention(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = torch.nn.Linear(4, 4, bias=False)
        self.k_proj = torch.nn.Linear(4, 2, bias=False)
        self.v_proj = torch.nn.Linear(4, 2, bias=False)

    def forward(self, x, past_key_value=None, *, use_cache=True):
        batch, new_len, _ = x.shape
        past_len = 0 if past_key_value is None else past_key_value[0].shape[1]
        positions, allowed = positions_and_mask(past_len, new_len, x.device)

        q = self.q_proj(x).reshape(batch, new_len, 2, 2)
        k_new = self.k_proj(x).reshape(batch, new_len, 1, 2)
        v_new = self.v_proj(x).reshape(batch, new_len, 1, 2)
        q, k_new = rotate_2d(q, positions), rotate_2d(k_new, positions)

        k_all, v_all, present = append_kv(k_new, v_new, past_key_value, use_cache=use_cache)

        # SDPA 使用 [B, H, T, D]，仅在计算时对齐 KV 头数。
        q = q.transpose(1, 2)
        k = k_all.transpose(1, 2).repeat_interleave(2, dim=1)
        v = v_all.transpose(1, 2).repeat_interleave(2, dim=1)
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
        return output.transpose(1, 2).reshape(batch, new_len, 4), present

torch.manual_seed(7)
layer = TinyAttention().eval()
x = torch.randn(1, 6, 4)

with torch.inference_mode():
    _, past = layer(x[:, :4])
    chunk_output, updated = layer(x[:, 4:], past)
    full_output, _ = layer(x, use_cache=False)

print("本轮输出", tuple(chunk_output.shape), "本轮返回的 K", tuple(updated[0].shape))
assert chunk_output.shape == (1, 2, 4)
assert updated[0].shape == (1, 6, 1, 2)
torch.testing.assert_close(updated[0][:, :4], past[0])
torch.testing.assert_close(chunk_output, full_output[:, 4:], atol=1e-6, rtol=1e-5)

固定输入与权重，**有缓存的续接输出应与整段重算的同位置输出一致**。如果 shape 正常但数值不一致，先查新位置偏移、mask 和旧 KV 的顺序。

## 两层怎么传状态？外层循环什么时候保存？

![Prefill 处理 A B C，选出 D。Decode 送入 D 后才追加 D 的 KV](figures/kv-cache-prefill-decode.svg)

图里用 `D` 和 `E` 示意两轮预测。下面的代码用 `range(max_new_tokens)` 控制生成步数，每轮实际从 logits 选出下一个 token。每轮把目前已经拿到的全部 token 记在 `input_ids`，但模型只接收 `input_ids[:, past_len:]`。模型向下逐层传 hidden state，为每一层单独传入旧 KV。每次 forward 结束后，生成循环接住返回的列表，覆盖自己的 `past_key_values`。

下方用两次 Attention 模拟两层。第一层的输出进入第二层，两层各自保存一份 KV。同一张向量表既用来查 embedding，也充当输出投影的权重。取最后一个位置的 hidden state，乘上转置后的向量表，得到形状为 `[B, vocab_size]` 的 logits。沿词表维取 `argmax`，得到 `[B, 1]` 的下一个 token ID。练习用的是随机权重，输出只用来对照缓存路径。

| 符号 | 含义 | 这个练习中的值 |
| --- | --- | --- |
| `B` | batch 中的序列数 | 1 |
| `T` | 序列轴上的 token 数，具体长度取决于张量 | `T_past` 是已缓存的长度，`T_new` 是本轮送入的长度，拼接后是 `T_total = T_past + T_new` |
| `H_q` / `H_kv` | Q 的头数 / K 和 V 各自的头数 | 2 / 1 |
| `D_model` | 每个 token 的 hidden state 维度 | 4 |
| `D_head` | 每个头里一个 Q、K 或 V 向量的维度 | 2 |

本例中，单个 token 的 hidden state 有 4 个分量，每个 KV 头的 K 和 V 各有 2 个分量。若把一个 token 的所有 K 头摊平，长度是 `H_kv × D_head`，V 也一样。MiniMind 布局下，每层分别保存 K 和 V 两个 `[B, T, H_kv, D_head]` 张量。

**先拆开缓存的三层结构。** 层号、K/V 的选择和张量的轴是三个不同的索引。

```text
past_key_values                  列表，共 L 项；本例 L = 2
├── [0] → (K₀, V₀)              第 0 层的二元组
│         ├── [0] → K₀          K 张量 [B, T_past, H_kv, D_head]
│         └── [1] → V₀          V 张量 [B, T_past, H_kv, D_head]
└── [1] → (K₁, V₁)              第 1 层的二元组，布局相同
```

所以 MiniMind 的 `past_key_values[0][0].shape[1]` 读作「取第 0 层 → 取 K → 读 K 的序列轴长度」。两个 `[0]` 都在取容器里的元素，只有最后的 `shape[1]` 在读张量的维度。这里每层处理了相同的位置数，查第 0 层的 K 就能知道处理进度。Hugging Face 示例把序列放在轴 2，读代码时要先确认布局。

具体到第 0 层的 K，`K₀[b, t, h, d]` 是第 `b` 条序列、位置 `t`、第 `h` 个 KV 头中的第 `d` 个分量。例如 `K₀[0, 2, 0, :]` 会取出第 0 条序列中 token `C` 的第 0 个 K 头，得到长度为 2 的向量。

以第二轮为例，`input_ids` 已包含 `A B C` 和首轮预测出的 token，形状是 `[1, 4]`。旧 KV 只覆盖 `A B C`，每个张量是 `[1, 3, 1, 2]`。本轮只送入那个刚选出的 token，所以 `input_slice` 是 `[1, 1]`，`hidden` 是 `[1, 1, 4]`，返回的每层 K/V 各是 `[1, 4, 1, 2]`。本轮预测出的下一个 token 会把 `input_ids` 增长到 `[1, 5]`，下轮才进入缓存。

In [ ]:
layers = [layer, TinyAttention().eval()]
token_vectors = x[0]  # [词表大小, D_model] = [6, 4]，A 到 F 各一行

def toy_forward(input_ids, past_key_values=None, *, use_cache=True):
    if past_key_values is None:
        past_key_values = [None] * len(layers)
    # 本函数的 input_ids 只含本轮输入，形状 [B, T_new]。
    hidden = token_vectors[input_ids]  # [B, T_new, D_model]
    presents = []  # 长度最终为层数 L，每层一对 KV 或 None
    for layer_index, attention in enumerate(layers):
        layer_past = past_key_values[layer_index]  # (K, V)，各 [B, T_past, H_kv, D_head]，或 None
        output, present = attention(hidden, layer_past, use_cache=use_cache)
        # output 是 [B, T_new, D_model]；present 的 K/V 各是 [B, T_total, H_kv, D_head]，或 None。
        hidden = hidden + output  # 残差连接，形状仍是 [B, T_new, D_model]
        presents.append(present)
    return hidden, presents

def run_loop(use_cache, max_new_tokens=3, eos_token_id=None):
    input_ids = torch.tensor([[0, 1, 2]])  # [B, T_input]，首轮是 [1, 3]（A B C）
    past_key_values = None  # 首轮没有历史；之后是 [(K₀, V₀), (K₁, V₁)]
    log = []
    step_logits = []  # 每轮一个 [B, vocab_size] 张量，用来对照有缓存和无缓存的结果
    with torch.inference_mode():
        for _ in range(max_new_tokens):
            if past_key_values is None:
                past_len = 0
            else:
                first_layer_k, _ = past_key_values[0]  # 取第 0 层的 (K, V)，这里只用 K
                # K 的四个轴依次是 [B, T_past, H_kv, D_head]。
                past_len = first_layer_k.shape[1]     # 序列轴长度 T_past
            input_slice = input_ids[:, past_len:]  # [B, T_new]，此处 T_new = T_input - past_len
            hidden, presents = toy_forward(input_slice, past_key_values, use_cache=use_cache)
            last_hidden = hidden[:, -1]  # [B, D_model]，只取最后一个位置
            logits = last_hidden @ token_vectors.T  # [B, vocab_size]，每个词表项一个分数
            next_token = logits.argmax(dim=-1, keepdim=True)  # [B, 1]，贪心解码
            step_logits.append(logits)

            # 1. 先追加本轮选出的 token，即使它是 EOS。
            input_ids = torch.cat((input_ids, next_token), dim=1)  # [B, T_input + 1]
            # 2. 保存本轮 forward 返回的 KV。它只覆盖已经送进模型的 token。
            past_key_values = presents if use_cache else None
            lengths = [] if past_key_values is None else [k.shape[1] for k, _ in past_key_values]
            predicted_id = next_token[0, 0].item()  # 这个练习只有一条序列（B=1）
            log.append((input_slice[0].tolist(), lengths, predicted_id))
            # 3. 最后检查 EOS。退出后不会再把刚追加的 EOS 喂给模型。
            if eos_token_id is not None and predicted_id == eos_token_id:
                break
    return input_ids, log, step_logits

cached_ids, cached_log, cached_logits = run_loop(use_cache=True, max_new_tokens=3)
full_ids, full_log, full_logits = run_loop(use_cache=False, max_new_tokens=3)
for new_ids, lengths, predicted_id in cached_log:
    print(f"本轮送入 {new_ids}，两层缓存长度 {lengths}，下一 token ID {predicted_id}")
print("最终序列", cached_ids[0].tolist())
print("关闭缓存时，各轮送入", [row[0] for row in full_log])

assert [len(new_ids) for new_ids, _, _ in cached_log] == [3, 1, 1]
assert [lengths for _, lengths, _ in cached_log] == [[3, 3], [4, 4], [5, 5]]
assert cached_ids.shape == (1, 6)
torch.testing.assert_close(cached_ids, full_ids)
torch.testing.assert_close(cached_logits, full_logits, atol=1e-6, rtol=1e-5)

# 把上面首轮预测出的 ID 当作教学用 EOS，稳定复现首轮提前停止。
demo_eos = cached_log[0][2]
stopped_ids, stopped_log, _ = run_loop(use_cache=True, max_new_tokens=3, eos_token_id=demo_eos)
assert stopped_ids.shape == (1, 4)
assert stopped_ids[0, -1].item() == demo_eos
assert stopped_log == [([0, 1, 2], [3, 3], demo_eos)]
print(f"遇到 EOS {demo_eos} 后停止，输出序列 {stopped_ids[0].tolist()}，每层 KV 长度仍为 3")

最后一轮结束后，序列有 6 个 token，但每层缓存只覆盖前 5 个。最后一个 token 刚由 logits 选出，只有继续执行下一轮才会写入它的 KV。`max_new_tokens` 是生成步数的上限。传入 `eos_token_id` 时，顺序是**追加新 token → 保存本轮 KV → 检查 EOS**。所以提前退出时，返回的原始 token 序列包含 EOS，而这轮保存的 KV 只覆盖已经做过 forward 的位置。这个练习只有一条序列。批量生成通常会为每条序列记录是否已结束，等所有序列都结束再退出整个循环。**缓存长度记录已经 forward 的位置数，`input_ids` 还可能包含刚选出的 token。**

缓存必须与它对应的输入前缀匹配。直接把旧缓存交给改过前文的请求，模型看到的仍是旧前文。跨请求匹配与近似复用见 [CacheBlend](CacheBlend.md)。

## 回到 MiniMind，按读写点跟源码

打开 [固定版本的 model_minimind.py](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py)。第一次读可以只跟下面几处，同时对照上面的 `run_loop`、`toy_forward` 和 `TinyAttention.forward`。

| 你刚写的步骤 | 在 MiniMind 里找 |
| --- | --- |
| 下一轮只送未处理的 token | [`generate` 的输入切片](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L269)。看 `past_len` 和传给 `forward` 的切片，先跳过采样参数。 |
| 确定新位置，把列表中的一对 KV 交给对应层 | [`MiniMindModel.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L214)。先看 `start_pos` 和 RoPE 切片，再看 `zip(self.layers, past_key_values)`。MiniMind 统一算位置，我们的练习在每层内算同一个偏移。 |
| 读取旧 K/V，追加本轮的新 K/V | [`Attention.forward` 的 RoPE 和拼接](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L117)。在 `torch.cat` 前后看 `xk.shape`，再看 `past_kv` 在 `repeat_kv` 前被构造。 |
| 把各层缓存收集到模型输出 | [`MiniMindBlock.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L191)返回一层的 `present_key_value`，模型逐层收集到 `presents`，再由 [`MiniMindForCausalLM.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L250)装进输出对象。 |
| 外层保管下一轮的状态 | [`generate` 的状态更新](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L285)。此时 `input_ids` 刚追加新选出的 token，`outputs.past_key_values` 只覆盖这一轮已经处理的位置。 |

**读源码自测。** 假设 prompt 长度是 4，刚采样了第一个新 token。下一轮调用 `forward` 前，写下 `input_ids.shape[1]`、`past_len` 和本轮输入长度。进入第 0 层后，再写出新 Q/K 的 RoPE 位置和 `torch.cat` 前后新 K 张量的长度。答案依次是 **5、4、1、位置 4、长度 1 → 5**。

## 换成 Transformers 的 Cache 对象

MiniMind 的 tuple 版本由调用方接住每层返回的状态。Hugging Face 常把整个缓存作为对象传入模型，Attention 通过 `cache.update(k_new, v_new, layer_idx)` 更新对应层，再用返回的 K/V 做 Attention。**对象版会修改传入的缓存对象**。重跑一条独立序列前，先重新初始化或重置它。

这个可选单元只操作缓存对象，需安装 Transformers。它复用第一节的张量，并把布局转成 Hugging Face 常用的 `[B, H_kv, T, D_head]`。`DynamicCache` 的序列轴随更新增长。`StaticCache` 的容量固定为 8，实际只写了 3 个位置，Attention 还要用 mask 排除未写入的槽位。你的 Transformers 5.3.0 在直接调用 StaticCache 时需要传写入位置。较新版本可以从对象内的有效长度推算。可以跟读 [Transformers 缓存策略](https://huggingface.co/docs/transformers/main/en/kv_cache)和 [`cache_utils.py`](https://github.com/huggingface/transformers/blob/main/src/transformers/cache_utils.py)。

In [ ]:
from transformers import DynamicCache, LlamaConfig, StaticCache

config = LlamaConfig(num_hidden_layers=1)
for cache in (DynamicCache(config=config), StaticCache(config=config, max_cache_len=8)):
    for start, k_new, v_new in ((0, k_prompt, v_prompt), (2, k_next, v_next)):
        positions = torch.arange(start, start + k_new.shape[1])
        keys, values = cache.update(
            k_new.transpose(1, 2), v_new.transpose(1, 2),
            layer_idx=0, cache_kwargs={"cache_position": positions},
        )
    valid = int(cache.get_seq_length())
    print(type(cache).__name__, "有效长度", valid, "存储形状", tuple(keys.shape))
    assert valid == 3
    torch.testing.assert_close(keys[:, :, :valid], k_all.transpose(1, 2))
    torch.testing.assert_close(values[:, :, :valid], v_all.transpose(1, 2))

在 vLLM 这样的服务端，“追加到序列尾部”会拆成**分配槽位、写入 KV、按块表读取**。读这类实现时也可以问同样的三个问题。新 KV 写到哪里？Attention 按什么顺序读？谁保管跨轮状态？从 [vLLM 块表](https://github.com/vllm-project/vllm/blob/main/vllm/v1/worker/block_table.py)和 [PagedAttention 论文](https://arxiv.org/abs/2309.06180)继续对照。

## KV cache 省了什么，显存又花在哪里

KV cache 的直接作用是**加速自回归解码**。本轮只给新 token 跑模型，历史位置的投影、Attention、归一化和 FFN 都可以跳过。全局 Attention 仍要用新 Q 读取历史 KV。已处理长度为 `T` 时，单 token Decode 的核心 Attention 随 `T` 线性增长。关闭缓存时，每轮通常要重新送入整个序列，核心 Attention 的乘加量随 `T²` 增长。第一次处理新提示词仍要做 Prefill，跨请求的前缀缓存才可能省掉已命中的部分。

| 显存花在哪 | 开启 KV cache | 关闭 KV cache |
| --- | --- | --- |
| 跨轮保留 | 各层保留已处理 token 的 KV | 上一轮的 KV 可以释放 |
| 本轮临时张量 | Decode 只推进新 token，但仍要读历史 KV | 要重新处理整个序列，临时张量通常更多 |

**关闭缓存省下的是跨轮保留的空间。** 每轮 forward 的峰值还取决于内核。朴素 Attention 可能显式构造很大的 `T × T` 分数矩阵，FlashAttention 可以分块计算，避免把整张分数矩阵写到显存。在推理里，各层的临时 K/V 也可以在用完后释放。反过来，开启缓存会让各层历史 KV 同时驻留。单看开关，无法判断哪个峰值更高。 [FlashAttention 论文](https://arxiv.org/abs/2205.14135)

### 固定一块卡，算能容纳多少 KV

假设 GPU 有 **24 GiB**，模型权重和运行空间合计先预留 **16 GiB**，只剩 **8 GiB** 给 KV。以 32 层、每头 128 维、BF16 KV 为例，比较 **8 个 KV 头**与**2 个 KV 头**。Q 头数和其他显存预算固定，方便单独看 KV 头数的影响。启用缓存省重算，GQA 等方法把每条请求的缓存做小。

每个已处理 token 的 KV 字节数是 `层数 × 2（K 和 V）× KV 头数 × 每头维度 × 元素字节数`。上下文预算里的 token 包含已处理的 prompt 和生成 token。这个估算暂不计分页取整、额外索引、前缀共享等因素。

In [ ]:
kib = 1024
gib = kib**3
kv_budget = (24 - 16) * gib
num_layers, head_dim, bytes_per_element = 32, 128, 2
tokens_per_request = 8192
resident_requests = 4

for h_kv in (8, 2):
    kv_bytes_per_token = num_layers * 2 * h_kv * head_dim * bytes_per_element
    token_slots = kv_budget // kv_bytes_per_token
    max_requests = token_slots // tokens_per_request
    max_tokens_per_request = token_slots // resident_requests
    print(
        f"{h_kv} 个 KV 头，每 token {kv_bytes_per_token // kib} KiB，"
        f"总槽位 {token_slots:,}。"
    )
    print(
        f"每条 8,192 token 时可容纳 {max_requests} 条；"
        f"固定 4 条时每条可分到 {max_tokens_per_request:,} token。"
    )

| 单独改变哪项 | 8 个 KV 头 | 2 个 KV 头 |
| --- | ---: | ---: |
| 每 token 的 KV | 128 KiB | 32 KiB |
| 8 GiB 能容纳的总 token 槽位 | 65,536 | 262,144 |
| 每条固定 8,192 token，能同时驻留几条 | 8 | 32 |
| 固定同时驻留 4 条，每条的缓存上限 | 16,384 token | 65,536 token |

两个 **4 倍**对应两种分配方式。固定每条长度，可以多驻留 **24 条**。固定并发为 4，每条可以多留 **49,152 个 token**。同一笔 8 GiB 预算要由并发和长度共同分摊。实际每条上下文还要取模型支持上限和显存容量上限两者中的较小值。

这也解释了为什么缓存可能比权重还大。按上面的 8 头配置，一条 131,072 token 的序列需要 **16 GiB KV**。作为对照，8B 参数的 BF16 权重约 **14.9 GiB**。这里是在比较大小，刚才只预留 8 GiB KV 的卡放不下这条序列。

### Batch 大小和吞吐分别怎么读

- **可驻留并发**是显存能同时保留多少条请求的状态。上表的 8 和 32 算的是这个上限。
- **Decode batch 大小**是某一步实际一起推进多少条序列。服务会调度请求进出，驻留的序列可以等待下一步，实际 batch 也会变化。
- **生成吞吐**是每秒一共产出多少个 token。简单同步 Decode 中，若每条序列每步产出 1 个 token，吞吐约为 **实际 batch ÷ 单步耗时（秒）**。例如 batch 为 8、每步 40 ms，吞吐是 200 token/s。batch 增到 16、每步也增到 80 ms，吞吐仍是 200 token/s。反过来，batch 仍是 8，单步降到 20 ms，吞吐就变成 400 token/s。这些数字只用来演示两者的关系。

**缓存做小，直接增加的是显存容量余量。** 这份余量可以拿去容纳更多请求或更长的历史。吞吐还取决于调度、每步耗时以及读 KV 的带宽。

## 动手改三处

每次只改一处，先猜结果再跑对应单元。每个实验结束后恢复原代码。

1. 把 `append_kv` 的拼接轴从 1 改成 2。用第一节的张量预测是拼接直接报错，还是缓存长度检查才会报错。
2. 只把 `TinyAttention` 中传给 `rotate_2d` 的位置改成从 0 开始，保留正确的 mask。第三节的 shape 断言还能通过，续接输出与整段重算的数值断言应失败。
3. 把循环里的 `input_slice` 改成完整的 `input_ids`，同时保留旧 KV。先算出第二轮缓存有多长、哪几个位置被重复写入，再看日志。